# D8.1 — Stage 2d Aggregate Result Analysis

Post-fire analytical adjudication of the Stage 2d 200-call batch (`batch_5cf76668-47d1-48d7-bd90-db06d31982ed`) against the pre-registered Scope Lock v2 claims in `docs/d7_stage2d/stage2d_expectations.md`.

This notebook is the first executable output of D8 (Stage 2d Result Adjudication). Downstream notebooks (D8.2, D8.3) consume its verdict.

## Scope, non-scope, hard rules

**Scope.** Gate adjudication for the five pre-registered claims (§6.2.1, §6.2.2, §6.3(a), §6.3(b), §6.4) and readout-only recording of §6.6 observation axes (alignment, SVR–alignment decouple, theme × label contingency, plausibility distribution).

**Non-scope.** §E3 Tier 1/2 bucket adjudication (D8.2). Axis-reversal narrative framing of §6.2.2 (D8.2 interpretation layer). Test-retest and drift magnitude analysis (D8.3). Any re-derivation of the fresh-7 subset at runtime.

**Hard rules (seven, binding on every cell below):**

1. Scoring universe is the 197 `critic_status == "ok"` records. Positions 42 and 87 (`d7b_error`) and position 116 (`skipped_source_invalid` synthetic pos 116) are quarantined, never imputed.
2. No imputation. A missing field, NaN, or absent key is a keyError — cells must raise, not substitute.
3. Pre-registered thresholds are honoured literally. No post-hoc reframing, threshold inversion, or direction swap.
4. §6.6 is readout-only — no PASS/FAIL adjudication on alignment, decouple, theme × label, plausibility, or neutral-stratum median.
5. The fresh-7 subset is the hard-coded literal `{3, 43, 68, 128, 173, 188, 198}`. Never re-derived from `deep_dive_candidates.json` or factor-set parsing.
6. All input artifacts are SHA256-verified against frozen anchors before any analytical cell executes.
7. Denominator semantics are explicit in every cell. 199 for §6.3 tail counts (UB full). 66 for §6.2.1 agreement. 5 for §6.2.2 divergence. 7 for §6.4 fresh-7. 197 for §6.6 observation readouts.

**Label-join invariants (two, inherited from Stage 2c):**

a. `universe_b_label` is the authoritative axis label for gate adjudication. `universe_a_label` is metadata only.
b. Per-call records are keyed by `candidate_position` except the synthetic pos 116, which uses `position` (Lock 1.5).

In [ ]:
# Cell 02 — imports, repo-root resolver, path pins, expected SHA anchors, helpers.

import hashlib
import json
from pathlib import Path

# Repo-root resolver: notebook lives at docs/test_notebooks/, walk up until we find
# a known repo-root sentinel (CLAUDE.md). Supports nbclient + direct Jupyter runs.
def _resolve_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise RuntimeError(f"repo root not found from cwd={here}")

REPO_ROOT = _resolve_repo_root()

BATCH_DIR = REPO_ROOT / "raw_payloads" / "batch_5cf76668-47d1-48d7-bd90-db06d31982ed"
CRITIC_DIR = BATCH_DIR / "critic"

AGGREGATE_PATH      = CRITIC_DIR / "stage2d_aggregate_record.json"
STARTUP_AUDIT_PATH  = CRITIC_DIR / "stage2d_startup_audit.json"
EXPECTATIONS_PATH   = REPO_ROOT / "docs" / "d7_stage2d" / "stage2d_expectations.md"
REPLAY_CANDIDATES_PATH = REPO_ROOT / "docs" / "d7_stage2d" / "replay_candidates.json"
SCOPE_LOCK_PATH     = REPO_ROOT / "docs" / "d7_stage2d" / "D7_STAGE2D_SCOPE_LOCK_v2.md"

EXPECTED_SHA256 = {
    "stage2d_aggregate_record.json": "09eeda3278c96ccf7b945c5edc9dde9bcfa51ca35138a63d36258514be5c323f",
    "stage2d_startup_audit.json":    "4f46de0902532c3d22d22d06c9f5604e82eff332b47f3aabf1603ef81e8a9ea5",
    "stage2d_expectations.md":       "98b87a702cc80df2d993d51857d4142f93f2ab8be66438bd2937c5dd374010a5",
    "replay_candidates.json":        "05706642cbeb5014d5072c172b343b01ee56d0eb8ee45afb2f3ce6e56665907e",
    "D7_STAGE2D_SCOPE_LOCK_v2.md":   "b4cad5873707c6eba272d313e0214011cb5ca91b142126013946f91a72496382",
}

# Hard denominators — asserted throughout the notebook.
STAGE2D_SOURCE_N = 200            # all per-call records
STAGE2D_LIVE_N   = 199            # live D7b attempted / non-skipped
STAGE2D_OK_N     = 197            # successfully scored (critic_status == "ok")
STAGE2D_D7B_ERR_N = 2             # pos 42, 87
STAGE2D_SKIPPED_N = 1             # pos 116 synthetic

ERROR_POSITIONS = (42, 87)
SKIPPED_POSITIONS = (116,)
QUARANTINE_POSITIONS = frozenset(ERROR_POSITIONS + SKIPPED_POSITIONS)

# Fresh-7 RSI-absent vol_regime — hard-coded literal per §6.4, hard rule 5.
FRESH_7_RSI_ABSENT = frozenset({3, 43, 68, 128, 173, 188, 198})
assert len(FRESH_7_RSI_ABSENT) == 7, "fresh-7 literal must be 7 positions"

# Stage 2b overlap positions — per expectations.md §E3 Tier 1 anchor.
STAGE2B_OVERLAP_POSITIONS = frozenset({17, 73, 74, 97, 138})

# Helpers.
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

checks: list[tuple[str, bool, str]] = []

def check(label: str, condition: bool, observed: str) -> None:
    """Append a PASS/FAIL row with observed value."""
    checks.append((label, bool(condition), observed))
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {label} -- observed={observed}")

def check_equal(label: str, observed, expected) -> None:
    """Append a PASS/FAIL row comparing observed to expected."""
    ok = observed == expected
    checks.append((label, ok, f"observed={observed!r} expected={expected!r}"))
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {label} -- observed={observed!r}; expected={expected!r}")

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"AGGREGATE_PATH exists: {AGGREGATE_PATH.exists()}")
print(f"5 input paths resolved; {len(EXPECTED_SHA256)} SHA anchors pinned.")


In [ ]:
# Cell 03 — SHA256 verification for aggregate + 4 secondary inputs.

for path in (
    AGGREGATE_PATH,
    STARTUP_AUDIT_PATH,
    EXPECTATIONS_PATH,
    REPLAY_CANDIDATES_PATH,
    SCOPE_LOCK_PATH,
):
    observed = sha256_of(path)
    expected = EXPECTED_SHA256[path.name]
    check_equal(f"sha256({path.name})", observed, expected)

# Hard failure if any SHA mismatches — adjudication is void without frozen inputs.
assert all(ok for _, ok, _ in checks), "SHA verification failed; adjudication aborted"


In [ ]:
# Cell 04 — load aggregate, validate 51-key shape, Lock 11 tail, status tally.

aggregate = load_json(AGGREGATE_PATH)

top_keys = list(aggregate.keys())
check_equal("aggregate top-level key count", len(top_keys), 51)
check_equal("aggregate Lock 11 tail key", top_keys[-1], "write_completed_at")

# Denominator surface on the aggregate itself.
check_equal("aggregate.stage2d_source_n",      aggregate["stage2d_source_n"],     STAGE2D_SOURCE_N)
check_equal("aggregate.stage2d_live_d7b_call_n", aggregate["stage2d_live_d7b_call_n"], STAGE2D_LIVE_N)
check_equal("aggregate.stage2d_skipped_positions", list(aggregate["stage2d_skipped_positions"]), list(SKIPPED_POSITIONS))
check_equal("aggregate.stage2b_overlap_positions (sorted)",
            sorted(aggregate["stage2b_overlap_positions"]),
            sorted(STAGE2B_OVERLAP_POSITIONS))

per_call_records = aggregate["per_call_records"]
check_equal("per_call_records length", len(per_call_records), STAGE2D_SOURCE_N)

critic_status_counts = aggregate["critic_status_counts"]
check_equal(
    "critic_status_counts partition",
    critic_status_counts,
    {"ok": STAGE2D_OK_N, "d7b_error": STAGE2D_D7B_ERR_N, "skipped_source_invalid": STAGE2D_SKIPPED_N},
)

d7b_scores_by_call = aggregate["d7b_scores_by_call"]
check_equal("d7b_scores_by_call entry count", len(d7b_scores_by_call), STAGE2D_SOURCE_N)

# Pos 42 / 87 / 116 must map to None in the scores dict (no SVR produced).
for pos in QUARANTINE_POSITIONS:
    check_equal(f"d7b_scores_by_call[str({pos})] is None", d7b_scores_by_call[str(pos)], None)


In [ ]:
# Cell 05 — build ok_frame (exactly 197 rows, critic_status == "ok").
# Root-level fields from per_call_records; score axes from d7b_scores_by_call[str(pos)].
# Schema note: d7b_scores_by_call preserves 200 positional slots, with
# quarantined records represented as None; the analytical scoring universe
# is constructed from per_call_records where critic_status == "ok".
# Hard rule 4: no try/except around dict lookups — KeyError is the correct failure mode.

ROOT_FIELDS = (
    "candidate_position",
    "candidate_theme",
    "pre_registered_label",
    "is_stage2b_overlap",
    "is_deep_dive_candidate",
    "test_retest_tier",
    "stratum_id",
    "prior_factor_sets_count",
    "theme_hint_factor_count",
    "firing_order",
    "call_index",
    "retry_count",
    "actual_cost_usd",
)

ok_frame: list[dict] = []
for rec in per_call_records:
    if rec["critic_status"] != "ok":
        continue
    position = rec["candidate_position"]
    scores = d7b_scores_by_call[str(position)]
    # scores is a dict with 3 axes; iteration order matters only for display.
    row = {field: rec[field] for field in ROOT_FIELDS}
    row["svr"]          = scores["structural_variant_risk"]
    row["alignment"]    = scores["semantic_theme_alignment"]
    row["plausibility"] = scores["semantic_plausibility"]
    ok_frame.append(row)

check_equal("ok_frame length", len(ok_frame), STAGE2D_OK_N)

ok_positions = {row["candidate_position"] for row in ok_frame}
check_equal("ok_frame unique positions count", len(ok_positions), STAGE2D_OK_N)

# Quarantine invariant: no error / skipped position is in ok_frame.
for pos in QUARANTINE_POSITIONS:
    check(f"pos {pos} absent from ok_frame", pos not in ok_positions, f"in_ok_frame={pos in ok_positions}")

# Label partition on ok_frame must match §6.2 denominators.
from collections import Counter
label_counts = Counter(row["pre_registered_label"] for row in ok_frame)
check_equal(
    "ok_frame pre_registered_label partition",
    dict(label_counts),
    {"agreement_expected": 64, "divergence_expected": 5, "neutral": 128},
)
# Note: agreement is 64 here (not 66) because pos 42 and 87 are agreement_expected
# candidates that quarantined as d7b_error. §6.2.1 gate denominator is 66 (universe),
# adjudicated in D8.1.4; the ok_frame itself is the 197-row scoring universe.


## §1 — Universe integrity

**Claim.** The Stage 2d batch returned 200 per-call records partitioning exactly as 197 `ok` + 2 `d7b_error` (pos 42, 87) + 1 `skipped_source_invalid` (pos 116 synthetic, Lock 1.5 17-key shape).

**Adjudication.** Assert the count partition literal and print the quarantine audit record for pos 42/87/116 (status code, error signature, raw payload path presence).

In [ ]:
# Cell 07 — universe integrity assertions + quarantine audit for pos 42 / 87 / 116.

# Restate the 200 -> 199 / 1 -> 197 / 2 partition.
check_equal("200 source = 199 live + 1 skipped",
            STAGE2D_LIVE_N + STAGE2D_SKIPPED_N, STAGE2D_SOURCE_N)
check_equal("199 live = 197 ok + 2 d7b_error",
            STAGE2D_OK_N + STAGE2D_D7B_ERR_N, STAGE2D_LIVE_N)
check_equal("200 source = 197 ok + 2 d7b_error + 1 skipped",
            STAGE2D_OK_N + STAGE2D_D7B_ERR_N + STAGE2D_SKIPPED_N, STAGE2D_SOURCE_N)

def _find_record(pos: int) -> dict:
    for rec in per_call_records:
        if rec.get("candidate_position") == pos or rec.get("position") == pos:
            return rec
    raise KeyError(f"record for position {pos} not found")

print()
print("=== Quarantine audit ===")
for pos in ERROR_POSITIONS:
    rec = _find_record(pos)
    check_equal(f"pos {pos} critic_status", rec["critic_status"], "d7b_error")
    raw_paths = rec.get("raw_payload_paths")
    assert isinstance(raw_paths, dict) and raw_paths, f"pos {pos} missing raw_payload_paths"
    print(
        f"  pos={pos}  status={rec['critic_status']}  "
        f"category={rec.get('d7b_error_category')!r}  "
        f"signature={rec.get('critic_error_signature')!r}  "
        f"retry_count={rec.get('retry_count')}  "
        f"raw_payload_paths_keys={sorted(raw_paths.keys())}"
    )

for pos in SKIPPED_POSITIONS:
    rec = _find_record(pos)
    check_equal(f"pos {pos} critic_status", rec["critic_status"], "skipped_source_invalid")
    check_equal(f"pos {pos} d7b_call_attempted", rec["d7b_call_attempted"], False)
    check_equal(f"pos {pos} d7b_error_category", rec["d7b_error_category"], "source_invalid")
    print(
        f"  pos={pos}  status={rec['critic_status']}  "
        f"skip_reason={rec.get('skip_reason')!r}"
    )

# Final integrity gate for Sections 0-1.
pass_count = sum(1 for _, ok, _ in checks if ok)
fail_count = sum(1 for _, ok, _ in checks if not ok)
print()
print(f"Sections 0-1 integrity checks: {pass_count} PASS, {fail_count} FAIL")
assert fail_count == 0, f"{fail_count} integrity checks failed"


## §2 — §6.2 Axis gates

**§6.2.1 Agreement axis (TBD-A1).** *Of the 66 UB agreement-labelled calls, ≥ 52 have SVR ≥ 0.5.* Operational pass = `count(svr >= 0.5) over {ub == "agreement_expected"} >= 52`.

**§6.2.2 Divergence axis (TBD-A2).** *Of the 5 UB divergence-labelled calls, ≥ 4 contradict the `divergence_expected` label with SVR ≥ 0.5.* Operational pass = `count(svr >= 0.5) over {ub == "divergence_expected"} >= 4`.

Adjudication is literal: observed count vs pre-registered threshold. PASS/FAIL only. Any interpretive framing (e.g. axis reversal) is D8.2 scope, not D8.1.

In [ ]:
# Cell 09 — §6.2.1 agreement axis gate.
# Placeholder only.

# agreement_universe = [r for r in ok_frame if r["universe_b_label"] == "agreement_expected"]
# assert len(agreement_universe) == 66, f"expected 66, got {len(agreement_universe)}"
# agreement_pass_count = sum(1 for r in agreement_universe if r["svr"] >= 0.5)
# AGREEMENT_THRESHOLD = 52
# agreement_verdict = "PASS" if agreement_pass_count >= AGREEMENT_THRESHOLD else "FAIL"
# print(f"§6.2.1 agreement: {agreement_pass_count}/66 at SVR≥0.5 vs threshold ≥{AGREEMENT_THRESHOLD} → {agreement_verdict}")


In [ ]:
# Cell 10 — §6.2.2 divergence axis gate.
# Literal pre-registration: ≥4 of 5 divergence_expected calls at SVR ≥ 0.5.
# No axis-reversal reframing here — D8.2 owns interpretation.
# Placeholder only.

# divergence_universe = [r for r in ok_frame if r["universe_b_label"] == "divergence_expected"]
# assert len(divergence_universe) == 5, f"expected 5, got {len(divergence_universe)}"
# divergence_pass_count = sum(1 for r in divergence_universe if r["svr"] >= 0.5)
# DIVERGENCE_THRESHOLD = 4
# divergence_verdict = "PASS" if divergence_pass_count >= DIVERGENCE_THRESHOLD else "FAIL"
# print(f"§6.2.2 divergence: {divergence_pass_count}/5 at SVR≥0.5 vs threshold ≥{DIVERGENCE_THRESHOLD} → {divergence_verdict}")


## §3 — §6.3 SVR distribution-shape gates

Two independent counts over the 199-call UB set (ok_frame plus anything with a numeric SVR; by construction = 197 ok records — the §6.3 denominator is **199** per expectations.md because pos 42 and pos 87 are UB members even though they lack an SVR score). The expectations doc states `over all 199 UB calls` — cell 12 honours that denominator exactly; d7b_error records contribute 0 to both tail counts (they have no SVR to compare).

**§6.3(a) upper tail.** ≥ 90 of 199 have SVR ≥ 0.80.

**§6.3(b) lower tail.** ≥ 30 of 199 have SVR ≤ 0.30.

Both must pass for §6.3 as a whole to pass.

In [ ]:
# Cell 12 — §6.3(a) upper-tail + §6.3(b) lower-tail gates.
# Denominator 199 honours expectations.md literally.
# Placeholder only.

# ub_universe = [r for r in per_call_records
#                if r.get("candidate_position") != 116  # synthetic pos 116 excluded from UB
#                and r["critic_status"] in {"ok", "d7b_error"}]
# assert len(ub_universe) == 199

# upper_tail_count = sum(1 for r in ub_universe if r.get("svr") is not None and r["svr"] >= 0.80)
# lower_tail_count = sum(1 for r in ub_universe if r.get("svr") is not None and r["svr"] <= 0.30)
# UPPER_THRESHOLD, LOWER_THRESHOLD = 90, 30
# upper_verdict = "PASS" if upper_tail_count >= UPPER_THRESHOLD else "FAIL"
# lower_verdict = "PASS" if lower_tail_count >= LOWER_THRESHOLD else "FAIL"
# joint_verdict  = "PASS" if (upper_verdict == lower_verdict == "PASS") else "FAIL"
# print(f"§6.3(a): {upper_tail_count}/199 at SVR≥0.80 vs ≥{UPPER_THRESHOLD} → {upper_verdict}")
# print(f"§6.3(b): {lower_tail_count}/199 at SVR≤0.30 vs ≥{LOWER_THRESHOLD} → {lower_verdict}")
# print(f"§6.3 joint: {joint_verdict}")


In [ ]:
# Cell 13 — SVR distribution readout: histogram bins + quartiles over 197 ok records.
# Readout only; not a gate.
# Placeholder only.

# svr_values = sorted(r["svr"] for r in ok_frame)
# # quartiles Q1/median/Q3 + min/max
# # histogram bins: [0.0,0.1), [0.1,0.2), ..., [0.9,1.0]


## §4 — §6.4 Fresh-7 RSI-absent vol_regime

**Claim.** Of the 7 fresh replay positions in the hard-coded fresh-7 subset `{3, 43, 68, 128, 173, 188, 198}`, **≥ 2 have SVR < 0.5**.

**Hard rule 5 binding.** The subset is the literal, pre-registered set — never re-derived from `deep_dive_candidates.json` or factor-set parsing. RSI-absence of the vol_regime is a pre-registered property of the stratum anchored in Stage 2c pos 138/143 evidence, not a per-call runtime check.

A placeholder code cell below records the RSI-absent + vol_regime-present property source (expectations.md §6.4 text + Stage 2c `call_0138_prompt.txt` / `call_0143_prompt.txt` references) as a provenance note for D8.2 audit — no runtime factor-set inspection is performed.

In [ ]:
# Cell 15 — fresh-7 literal set + RSI-absence provenance source check.
# Placeholder only. No runtime factor-set re-derivation (hard rule 5).

# FRESH_7 = frozenset({3, 43, 68, 128, 173, 188, 198})
# assert len(FRESH_7) == 7

# fresh7_records = [r for r in ok_frame if r["candidate_position"] in FRESH_7]
# # pos 188 has factor_set_prior_occurrences >= 1 per §6.4 edge-case note; still in denom.
# assert len(fresh7_records) == 7, f"fresh-7 coverage incomplete: {len(fresh7_records)}/7"

# # RSI-absence provenance: text anchor only; no runtime check.
# RSI_ABSENCE_SOURCE = (
#     "expectations.md §6.4 + Stage 2c call_0138_prompt.txt / call_0143_prompt.txt"
# )


In [ ]:
# Cell 16 — fresh-7 per-position table: SVR, alignment, UB label.
# Computes §6.4 gate: count(SVR < 0.5) >= 2.
# Placeholder only.

# rows = [(r["candidate_position"], r["svr"], r.get("alignment"), r["universe_b_label"])
#         for r in sorted(fresh7_records, key=lambda x: x["candidate_position"])]
# fresh7_low_count = sum(1 for _, svr, *_ in rows if svr < 0.5)
# FRESH7_THRESHOLD = 2
# fresh7_verdict = "PASS" if fresh7_low_count >= FRESH7_THRESHOLD else "FAIL"
# print(f"§6.4 fresh-7: {fresh7_low_count}/7 at SVR<0.5 vs ≥{FRESH7_THRESHOLD} → {fresh7_verdict}")


## §5 — §6.6 Readout (observation axes only)

Per expectations.md §6.6, these axes are **recorded, not pre-registered** — no PASS/FAIL adjudication. Hard rule 4 binding.

1. **Alignment distribution** — mean, quartiles, range across 199 calls; per-axis (agreement / divergence / neutral) within UB.
2. **SVR–alignment decoupling** — counts of `svr >= 0.75 AND alignment <= 0.50` and the inverse `svr <= 0.25 AND alignment >= 0.75`. Stage 2c anchor: pos 102 (SVR 0.95 / aln 0.40) — sole decouple; no pre-registered count.
3. **Theme × label contingency** — 6 themes × 3 UB labels = 18-cell count table.
4. **Plausibility score distribution** — parallel to (1).

The neutral-stratum SVR median reported in cell 21 is observation only. Disk verification of expectations.md confirms **no numeric bound `[0.45, 0.70]` is pre-registered for Stage 2d** — the only `median` reference in expectations.md (L584) is a per-candidate MODERATE-LOW bucket test, not an aggregate neutral-median bound.

In [ ]:
# Cell 18 — alignment distribution readout: overall + per-UB-axis.
# Readout only; not a gate.
# Placeholder only.

# alignment_all = [r["alignment"] for r in ok_frame if r.get("alignment") is not None]
# # mean, Q1, median, Q3, min, max
# # repeat per UB axis: agreement_expected / divergence_expected / neutral


In [ ]:
# Cell 19 — SVR–alignment decoupling counts.
# Readout only.
# Placeholder only.

# hi_lo = sum(1 for r in ok_frame if r["svr"] >= 0.75 and r.get("alignment", 1.0) <= 0.50)
# lo_hi = sum(1 for r in ok_frame if r["svr"] <= 0.25 and r.get("alignment", 0.0) >= 0.75)
# # list per-position rows for each decouple direction


In [ ]:
# Cell 20 — 6-theme × 3-label contingency table (18 cells).
# Readout only.
# Placeholder only.

# THEMES = [...]  # 6-theme canonical list from D6
# UB_LABELS = ["agreement_expected", "divergence_expected", "neutral"]
# # nested Counter over (theme, ub_label)


In [ ]:
# Cell 21 — neutral-stratum SVR median + IQR. OBSERVATION ONLY.
# No pre-registered bound; disk verified no [0.45, 0.70] anchor.
# Placeholder only.

# neutral_svr = sorted(r["svr"] for r in ok_frame if r["universe_b_label"] == "neutral")
# # Q1 / median / Q3 + n
# # plausibility distribution can follow as a short readout here or be its own cell


## §6 — Stratum / theme cross-tabs (forensic observation)

Forensic readout of theme × UB label × SVR bucket. Per Lock 3.6 and §6.5 ruling, no theme-stratified threshold is pre-registered; this section is recorded for D8.2 interpretive input and carries no pass/fail.

Per Q3 adjudication (Round 2 ratification): report direct counts and within-row percentages only. **No binomial p-value. No Wilson CI. No hypothesis-test framing.** Columns: theme, UB label, SVR bucket, count, row denominator, within-row percentage.

In [ ]:
# Cell 23 — theme × UB label × SVR bucket cross-tab.
# Direct counts + within-row percentage ONLY. No CI. No p-value.
# Placeholder only.

# SVR_BUCKETS = ["LOW (<0.25)", "MOD-LOW [0.25,0.50)", "MOD-HIGH [0.50,0.85)", "HIGH [0.85,1.0]"]
# rows: theme, ub_label, svr_bucket, count, row_denominator, within_row_pct


## §7 — Synthesis

Assemble the per-claim gate adjudication table, the research implication matrix, and the final D8.1 verdict. Downstream D8.2 consumes this section's summary artifacts.

In [ ]:
# Cell 25 — per-claim PASS/FAIL summary table.
# Placeholder only.

# summary = [
#     {"claim": "§6.2.1 agreement",       "observed": ..., "threshold": "≥52/66 at SVR≥0.5", "verdict": ...},
#     {"claim": "§6.2.2 divergence",      "observed": ..., "threshold": "≥4/5 at SVR≥0.5",  "verdict": ...},
#     {"claim": "§6.3(a) upper tail",     "observed": ..., "threshold": "≥90/199 at SVR≥0.80", "verdict": ...},
#     {"claim": "§6.3(b) lower tail",     "observed": ..., "threshold": "≥30/199 at SVR≤0.30", "verdict": ...},
#     {"claim": "§6.4 fresh-7",           "observed": ..., "threshold": "≥2/7 at SVR<0.5",  "verdict": ...},
# ]
# render as table


## §7.1 — Research implication matrix

Seven rows, claim-per-row. Each row names a finding, the observed result, its pre-registered status (gate vs readout), and the D8 follow-up implication. No scenario-combinatoric rows.

| # | finding | observed_result | pre_registered_status | d8_followup |
|---|---|---|---|---|
| 1 | §6.2.1 agreement axis | _filled after cell 09 runs_ | gate | carry verdict into D8.2 axis narrative |
| 2 | §6.2.2 divergence axis | _filled after cell 10 runs_ | gate | if FAIL, D8.2 owns axis-reversal interpretation question |
| 3 | §6.3(a) upper-tail distribution | _filled after cell 12 runs_ | gate | carry into D8.2 distributional-shape narrative |
| 4 | §6.3(b) lower-tail distribution | _filled after cell 12 runs_ | gate | carry into D8.2 distributional-shape narrative |
| 5 | §6.4 fresh-7 RSI-absent vol_regime | _filled after cell 16 runs_ | gate | feeds D8.3 test-retest context for pos 138/143 |
| 6 | §6.6 observation axes (alignment / decouple / theme × label / plausibility / neutral median) | _filled after cells 18–21 run_ | readout | recorded as forensic input to D8.2; no gate adjudication |
| 7 | D8.2 / D8.3 carry-forward | aggregated from rows 1–6 | n/a | §E3 Tier 1/2 bucket adjudication → D8.2; test-retest + drift → D8.3 |

## §7.2 — Final D8.1 analytical verdict

Final D8.1 analytical verdict will be computed after Sections 2–7 execute.

Gate claims are §6.2.1, §6.2.2, §6.3(a), §6.3(b), and §6.4.

§6.6 is readout-only.